# Week 76, Day 1: How the web works

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w076d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from urllib.parse import urlsplit

url = urlsplit("https://atlas.example.com:8443/courses/python/w076?day=1&lang=en#goals")
print(url.scheme, url.hostname, url.port)
print(url.path, url.query, url.fragment)

In [ ]:
import ipaddress

for text in ["192.168.1.20", "8.8.8.8", "127.0.0.1", "10.4.0.9", "2606:4700::1111"]:
    address = ipaddress.ip_address(text)
    print(f"{text:>16}  IPv{address.version}  private={address.is_private}  loopback={address.is_loopback}")

office = ipaddress.ip_network("192.168.1.0/24")
print(office.num_addresses, ipaddress.ip_address("192.168.1.200") in office, ipaddress.ip_address("192.168.2.1") in office)
print(list(ipaddress.ip_network("10.0.0.0/30").hosts()))

In [ ]:
records = {
    "www.example.com": [("CNAME", "web.example.net", 300)],
    "web.example.net": [("A", "203.0.113.10", 60), ("A", "203.0.113.11", 60)],
    "example.com": [("A", "203.0.113.5", 3600), ("MX", "mail.example.com", 3600)],
}

def resolve(name, kind="A"):
    seen = []
    while True:
        seen.append(name)
        answers = records.get(name, [])
        alias = [value for k, value, ttl in answers if k == "CNAME"]
        if alias:
            name = alias[0]
            continue
        return seen, [value for k, value, ttl in answers if k == kind]

print(resolve("www.example.com"))
print(resolve("example.com", "MX"))

In [ ]:
from urllib.parse import urlencode, urljoin, parse_qs, urlsplit, quote

query = urlencode({"q": "fish & chips", "page": 2, "lang": "en-GB"})
url = f"https://search.example.com/find?{query}"
print(url)
print(parse_qs(urlsplit(url).query))
print(urljoin("https://atlas.example.com/w076/d1.html", "d2.html"), urljoin("https://atlas.example.com/w076/d1.html", "/index.html"))
print(quote("week 76/notes.txt"), quote("week 76/notes.txt", safe=""))

def request_time(latency_ms, round_trips, size_bytes, bandwidth_mbps):
    return latency_ms * round_trips + size_bytes * 8 / (bandwidth_mbps * 1000)

print(round(request_time(20, 3, 500_000, 50), 1), "ms nearby;", round(request_time(150, 3, 500_000, 50), 1), "ms far away")

> **Key idea.** The internet is layered: link (local frames), IP (packets between any addresses, routed by longest prefix, no promises), TCP (a reliable ordered byte stream, after a one-round-trip handshake) or UDP (bare datagrams), TLS (encryption and certificates), and application protocols such as HTTP; each wraps the one above in its header. IPv4 is 32 bits, IPv6 128; CIDR networks share a prefix; private and loopback ranges are special; ports pick a program (80, 443, 22, 53). DNS resolves names through resolvers, root, TLD and authoritative servers, returning A, AAAA, CNAME, MX and TXT records cached for their TTL. urllib.parse splits, joins and encodes URLs (urlencode, quote, urljoin). Request time is mostly round trips times latency, plus size over bandwidth, so reuse connections.

### Exercise: Taking URLs apart

Write `describe(url)`: a dict with `"scheme"`, `"host"` (lower case), `"port"` (the explicit port, or the default: 80 for `http`, 443 for `https`; `None` for other schemes without one), `"path"` (`"/"` if empty), `"query"` (a dict from `parse_qs`) and `"secure"` (whether the scheme is `https`). Then `same_origin(a, b)`: whether two URLs have the same scheme, host and port (after filling in defaults).

In [ ]:
from urllib.parse import urlsplit, parse_qs

DEFAULT_PORTS = {"http": 80, "https": 443}

def describe(url):
    ___

def same_origin(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_urls():\n    \"urlsplit and origins\"\n    d = describe(\"https://Atlas.Example.com/w076?day=1&tag=net&tag=http#goals\")\n    assert d == {\"scheme\": \"https\", \"host\": \"atlas.example.com\", \"port\": 443, \"path\": \"/w076\", \"query\": {\"day\": [\"1\"], \"tag\": [\"net\", \"http\"]}, \"secure\": True}\n    assert describe(\"http://localhost:8000\")[\"port\"] == 8000 and describe(\"http://localhost:8000\")[\"path\"] == \"/\"\n    assert describe(\"ftp://files.example.com/a\")[\"port\"] is None\n    assert same_origin(\"https://a.com/x\", \"https://A.com:443/y?z=1\") and not same_origin(\"https://a.com\", \"http://a.com\")\n    assert not same_origin(\"https://a.com\", \"https://api.a.com\") and not same_origin(\"http://a.com:8000\", \"http://a.com\")")

### Exercise: Addresses and networks

Write `classify(text)`: for an IP address (v4 or v6), return `"loopback"`, `"link-local"`, `"private"` or `"public"`, checking in that order with the `ipaddress` properties `is_loopback`, `is_link_local` and `is_private` (link-local before private, because Python counts link-local addresses as private too), and `"invalid"` if the text isn't an address (`ValueError`). Then `group_by_network(addresses, networks)`: a dict from each network (as given, a CIDR string) to the sorted list of the addresses inside it, plus `"other"` for addresses in none of them; when networks overlap, an address belongs to the one with the **longest** prefix.

In [ ]:
import ipaddress

def classify(text):
    ___

def group_by_network(addresses, networks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_addresses():\n    \"ipaddress\"\n    assert [classify(a) for a in [\"127.0.0.1\", \"10.1.2.3\", \"192.168.0.10\", \"169.254.1.1\", \"8.8.8.8\", \"::1\", \"fe80::1\", \"2606:4700::1111\", \"300.1.1.1\", \"nope\"]] == \\\n        [\"loopback\", \"private\", \"private\", \"link-local\", \"public\", \"loopback\", \"link-local\", \"public\", \"invalid\", \"invalid\"]\n    groups = group_by_network([\"10.0.0.5\", \"10.0.1.7\", \"10.1.0.1\", \"192.168.1.9\", \"8.8.4.4\"], [\"10.0.0.0/8\", \"10.0.1.0/24\", \"192.168.1.0/24\"])\n    assert groups == {\"10.0.0.0/8\": [\"10.0.0.5\", \"10.1.0.1\"], \"10.0.1.0/24\": [\"10.0.1.7\"], \"192.168.1.0/24\": [\"192.168.1.9\"], \"other\": [\"8.8.4.4\"]}")

### Exercise: A caching resolver

Write a class `Resolver(records, clock)`: `records` maps a name to a list of `(kind, value, ttl)`; `clock()` returns the current time in seconds. `resolve(name, kind="A")` follows `CNAME` records (at most 8 steps; more, or a loop, raises `ValueError("CNAME loop")`) and returns the sorted list of values of that kind for the final name (empty if none; names are case-insensitive, so lower-case them). Each name looked up is **cached** with its records until the smallest TTL among them runs out; `lookups` counts how many times the resolver had to read `records` itself (a cache miss), so repeated questions within the TTL cost nothing.

In [ ]:
class Resolver:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_resolver():\n    \"aliases, caching and TTLs\"\n    records = {\n        \"www.example.com\": [(\"CNAME\", \"web.example.net\", 300)],\n        \"web.example.net\": [(\"A\", \"203.0.113.11\", 60), (\"A\", \"203.0.113.10\", 60), (\"AAAA\", \"2001:db8::1\", 60)],\n        \"loop.example.com\": [(\"CNAME\", \"loop2.example.com\", 60)],\n        \"loop2.example.com\": [(\"CNAME\", \"loop.example.com\", 60)],\n    }\n    now = [0]\n    r = Resolver(records, lambda: now[0])\n    assert r.resolve(\"WWW.example.com\") == [\"203.0.113.10\", \"203.0.113.11\"] and r.lookups == 2\n    assert r.resolve(\"www.example.com\", \"AAAA\") == [\"2001:db8::1\"] and r.lookups == 2\n    now[0] = 61\n    assert r.resolve(\"www.example.com\") == [\"203.0.113.10\", \"203.0.113.11\"] and r.lookups == 3\n    assert r.resolve(\"missing.example.com\") == []\n    try:\n        r.resolve(\"loop.example.com\")\n        assert False\n    except ValueError as e:\n        assert str(e) == \"CNAME loop\"")

### Exercise: Where the time goes

Write `fetch_time(latency_ms, bandwidth_mbps, size_bytes, new_connection=True, https=True, dns_cached=True)`: the time in milliseconds to fetch one resource: one round trip for DNS if not cached, one for the TCP handshake and one for the TLS handshake if it's a new connection (TLS only for HTTPS), one for the request and response, plus the transfer time (`size_bytes * 8 / (bandwidth_mbps * 1000)` ms). Then `page_time(latency_ms, bandwidth_mbps, sizes, keep_alive=True)`: the total for fetching resources one after another from one HTTPS server: the first needs DNS and a new connection; later ones reuse the connection if `keep_alive`, or need a new connection each (but DNS stays cached). Round results to one decimal place.

In [ ]:
def fetch_time(latency_ms, bandwidth_mbps, size_bytes, new_connection=True, https=True, dns_cached=True):
    ___

def page_time(latency_ms, bandwidth_mbps, sizes, keep_alive=True):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timing():\n    \"round trips and bandwidth\"\n    assert fetch_time(50, 10, 125_000) == 250.0\n    assert fetch_time(50, 10, 125_000, dns_cached=False) == 300.0 and fetch_time(50, 10, 125_000, https=False) == 200.0\n    assert fetch_time(50, 10, 125_000, new_connection=False) == 150.0\n    sizes = [50_000, 10_000, 10_000, 10_000]\n    assert page_time(100, 20, sizes) == 732.0 and page_time(100, 20, sizes, keep_alive=False) == 1332.0")

### Exercise: Bytes in segments

TCP numbers every byte. Write `segment(data, size, start=0)`: split `data` (bytes) into a list of `(seq, chunk)` segments of at most `size` bytes, where `seq` is the number of the chunk's first byte, counting from `start`. Then `reassemble(segments, start=0)`: put segments back together whatever order they arrive in, ignoring duplicates (and overlapping retransmissions), and return `(data, missing)`: the bytes received **in order** from `start` up to the first gap, and the sequence number of the first missing byte (or `None` if there's no gap after the data received).

In [ ]:
def segment(data, size, start=0):
    ___

def reassemble(segments, start=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_segments():\n    \"ordering, duplicates and gaps\"\n    data = b\"Hello, network!\"\n    segs = segment(data, 4, start=1000)\n    assert segs == [(1000, b\"Hell\"), (1004, b\"o, n\"), (1008, b\"etwo\"), (1012, b\"rk!\")]\n    shuffled = [segs[2], segs[0], segs[3], segs[0], segs[1]]\n    assert reassemble(shuffled, start=1000) == (data, None)\n    assert reassemble([segs[0], segs[2]], start=1000) == (b\"Hell\", 1004)\n    assert reassemble([(1002, b\"llo, \")] + segs[:1], start=1000) == (b\"Hello, \", None)\n    assert reassemble([], start=5) == (b\"\", None)")

### Exercise: Building URLs safely

Write `search_url(base, **params)`: `base` plus `?` and the parameters encoded with `urlencode` (keeping their order, skipping any whose value is `None`, and encoding list values as repeated keys). Then `file_url(base, *segments)`: join path segments onto `base` (which ends with `/`), percent-encoding each segment completely (including `/`, with `quote(segment, safe="")`), so a segment can never add extra path levels.

In [ ]:
from urllib.parse import urlencode, quote

def search_url(base, **params):
    ___

def file_url(base, *segments):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_building():\n    \"encoding, not joining strings\"\n    from urllib.parse import urlsplit, parse_qs\n    url = search_url(\"https://shop.example.com/search\", q=\"fish & chips\", page=2, sort=None, tag=[\"hot\", \"cheap\"])\n    assert url == \"https://shop.example.com/search?q=fish+%26+chips&page=2&tag=hot&tag=cheap\"\n    assert parse_qs(urlsplit(url).query) == {\"q\": [\"fish & chips\"], \"page\": [\"2\"], \"tag\": [\"hot\", \"cheap\"]}\n    assert search_url(\"https://a.com/\") == \"https://a.com/\"\n    assert file_url(\"https://files.example.com/users/\", \"ada\", \"notes 2026.txt\") == \"https://files.example.com/users/ada/notes%202026.txt\"\n    assert file_url(\"https://files.example.com/users/\", \"../admin\") == \"https://files.example.com/users/..%2Fadmin\"")

### Bug hunt: Addresses sorted as text

The addresses are sorted as strings, so `10.0.0.10` comes before `10.0.0.9` and this prints `['10.0.0.1', '10.0.0.10', '10.0.0.9']`. Sort them as addresses, so it prints `['10.0.0.1', '10.0.0.9', '10.0.0.10']`.

In [ ]:
import ipaddress

addresses = ["10.0.0.10", "10.0.0.1", "10.0.0.9"]
print(sorted(addresses))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"numeric order\"\n    expect_output(\"['10.0.0.1', '10.0.0.9', '10.0.0.10']\")")

### Bug hunt: A query string that split in two

The query is built by joining strings, so the `&` inside "fish & chips" starts a new parameter and the server sees the search as just `fish `: this prints `{'q': ['fish '], ' chips': ['']}`. Build it with `urlencode`, so this prints `{'q': ['fish & chips']}`.

In [ ]:
from urllib.parse import urlsplit, parse_qs

term = "fish & chips"
url = "https://shop.example.com/search?q=" + term
print(parse_qs(urlsplit(url).query, keep_blank_values=True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"encoded values\"\n    expect_output(\"{'q': ['fish & chips']}\")")

### Bug hunt: A base without its slash

`urljoin` treats the last part of a path without a trailing slash as a **file**, so joining `"v2"` onto `https://api.example.com/docs` replaces `docs` and this prints `https://api.example.com/v2`. Give the base its trailing slash so it prints `https://api.example.com/docs/v2`.

In [ ]:
from urllib.parse import urljoin

base = "https://api.example.com/docs"
print(urljoin(base, "v2"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"folders end with a slash\"\n    expect_output(\"https://api.example.com/docs/v2\")")

### Puzzle: What happens when you visit a page

Arrange the steps of fetching `https://atlas.example.com/` on a fresh connection, in order. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of addresses in 192.168.0.0/22
q = ___   # the default port for https
r = ___   # the DNS record type that makes a name an alias for another
s = ___   # True or False: TCP guarantees that bytes arrive in order
t = ___   # urljoin("https://a.com/x/y.html", "../z.html")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 1024 and q == 443 and r == \"CNAME\", \"p, q and r\"\n    assert s is True and t == \"https://a.com/z.html\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a routing table

Routers forward each packet by the **longest matching prefix**. Write a class `RoutingTable`: `add(network, next_hop, metric=1)` adds a route (a CIDR string and where to send matching packets; adding the same network again keeps the route with the **lower** metric); `remove(network)`; `route(address)` returns the next hop for an address, choosing the matching route with the longest prefix, then the lowest metric, then the earliest added, or `None` if nothing matches (a default route is `0.0.0.0/0`); and `summary()` returns the routes as `"NETWORK via NEXT_HOP (metric M)"` strings, sorted by network address and then prefix length. Networks must be valid (`ValueError` otherwise; use `ipaddress.ip_network` with its default strict check, so `10.0.0.1/8` is refused).

In [ ]:
import ipaddress

class RoutingTable:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_routing():\n    \"longest prefix match\"\n    t = RoutingTable()\n    t.add(\"0.0.0.0/0\", \"isp\")\n    t.add(\"10.0.0.0/8\", \"core\")\n    t.add(\"10.1.0.0/16\", \"branch\", metric=5)\n    t.add(\"10.1.0.0/16\", \"backup\", metric=2)\n    t.add(\"10.1.2.0/24\", \"lab\")\n    assert t.route(\"10.1.2.3\") == \"lab\" and t.route(\"10.1.9.9\") == \"backup\" and t.route(\"10.200.0.1\") == \"core\"\n    assert t.route(\"8.8.8.8\") == \"isp\"\n    t.add(\"10.1.0.0/16\", \"worse\", metric=9)\n    assert t.route(\"10.1.9.9\") == \"backup\"\n    t.remove(\"10.1.2.0/24\")\n    assert t.route(\"10.1.2.3\") == \"backup\"\n    assert t.summary() == [\"0.0.0.0/0 via isp (metric 1)\", \"10.0.0.0/8 via core (metric 1)\", \"10.1.0.0/16 via backup (metric 2)\"]\n    t.remove(\"0.0.0.0/0\")\n    assert t.route(\"8.8.8.8\") is None\n    for bad in [\"10.0.0.1/8\", \"300.0.0.0/8\", \"nonsense\"]:\n        try:\n            t.add(bad, \"x\")\n            assert False, bad\n        except ValueError:\n            pass")

## Warm-up

Every part of a URL is a decision about the trip: `https` chooses the protocol (HTTP over an encrypted connection), the host name must be looked up to find a computer, the port picks which program on that computer answers, and the path and query say what to ask it for. The fragment (`#goals`) never leaves your browser at all.

## Layers

The internet is built in **layers**, each using the one below and hiding its details from the one above:

- **Link** (Ethernet, Wi-Fi): moves frames between machines on the **same** network.
- **Internet (IP)**: moves **packets** between any two machines, across many networks, by **IP address**. Each **router** on the way looks at the destination and forwards the packet one hop closer. IP makes no promises: packets can be lost, duplicated or arrive out of order.
- **Transport (TCP or UDP)**: **TCP** turns unreliable packets into a reliable, ordered **stream of bytes** between two programs: it numbers the bytes, acknowledges what arrives, resends what's lost, reorders, and slows down when the network is congested. It starts with a three-way **handshake** (SYN, SYN-ACK, ACK) costing one round trip. **UDP** just sends individual datagrams, with no guarantees, for when speed matters more than completeness (games, video calls, DNS).
- **Security (TLS)**: encrypts the stream and proves the server's identity with a **certificate**. Its handshake costs another round trip (one with TLS 1.3).
- **Application (HTTP, DNS, SMTP...)**: the language the programs speak. HTTP (tomorrow) is requests and responses carried on TCP (and TLS); the newest version, HTTP/3, runs over **QUIC**, which is built on UDP and does the reliability itself.

Each layer wraps the data from the layer above in a **header** of its own (encapsulation): an HTTP request inside TLS records inside TCP segments inside IP packets inside Wi-Fi frames.

## Addresses and ports

An **IP address** identifies a network interface. **IPv4** addresses are 32 bits, written as four bytes: `93.184.215.14`. There are only about four billion, so **IPv6** uses 128 bits, written in hexadecimal groups: `2606:2800:21f:cb07:6820:80da:af6b:8b2c` (with `::` standing for a run of zero groups).

- A **network** (or subnet) is a range of addresses sharing a prefix, written in **CIDR** notation: `192.168.1.0/24` is the 256 addresses whose first 24 bits match. Routers forward by prefix, choosing the **longest** matching one.
- Some ranges are special: **private** addresses (`10.0.0.0/8`, `172.16.0.0/12`, `192.168.0.0/16`) are used inside homes and offices, behind a router that translates them (**NAT**); **loopback** (`127.0.0.1`, `::1`) is your own machine.
- A **port** (0 to 65535) picks one program on a machine: a connection is identified by both ends' address **and** port. Well-known ports: 80 for HTTP, 443 for HTTPS, 22 for SSH, 53 for DNS, 5432 for PostgreSQL. Programs you run while developing usually use high ports such as 8000.

Python's `ipaddress` module understands all of this.

## DNS

People use names; IP uses numbers. The **Domain Name System** translates. Names are a tree, read from the right: `atlas.example.com.` sits under `example.com`, under `com`, under the root.

- To resolve a name, your computer asks a **recursive resolver** (run by your network or a public service). If it doesn't already know, it asks a **root** server, which refers it to the servers for `com`, which refer it to the **authoritative** servers for `example.com`, which give the answer.
- Answers are **records**: `A` (an IPv4 address), `AAAA` (IPv6), `CNAME` (this name is an alias: look up that other name instead), `MX` (mail servers), `TXT` (text, used for verification), `NS` (which servers are authoritative).
- Every record has a **TTL** (time to live) in seconds: resolvers and computers **cache** answers for that long, which is why DNS is fast, and why a changed record can take a while to be seen everywhere.

## URLs, and what takes the time

A **URL** names a resource and how to reach it: `scheme://user@host:port/path?query#fragment`. `urllib.parse` handles them:

- **`urlsplit`** takes one apart (`.scheme`, `.hostname`, `.port`, `.path`, `.query`, `.fragment`); `urlunsplit` puts one back together.
- **`parse_qs`** / `parse_qsl` decode a query string; **`urlencode`** builds one, **percent-encoding** characters that would otherwise be misread: a space becomes `+` (or `%20`), `&` becomes `%26`. **`quote`** encodes a path segment. Never build a query by joining strings: a value containing `&` or `=` would break it.
- **`urljoin(base, link)`** resolves a relative link the way a browser does: `urljoin("https://a.com/docs/intro", "setup")` is `https://a.com/docs/setup`.
- An **origin** is the scheme, host and port together; browsers use it for their security rules (Week 77).

**What takes the time.** A request's time is mostly **round trips**, each costing the **latency** (the time for a message to cross the network and back: a few milliseconds nearby, 100 ms or more across the world), plus the time to push the bytes through the **bandwidth** (the capacity, in megabits per second). A fresh HTTPS request needs a DNS lookup (if not cached), a TCP handshake (one round trip), a TLS handshake (one more with TLS 1.3), and then the request itself (one more). That's why connections are **reused** (keep-alive), why far-away servers feel slow even on fast lines, and why content delivery networks put copies of files close to users.

## Practice

## On your own computer

The real network is a terminal away:

```bash
python -c "import socket; print(socket.getaddrinfo('python.org', 443, proto=6)[0][4])"
```

```bash
nslookup python.org
```

```bash
traceroute python.org
```

(`tracert` on Windows.) The first resolves a name with your system's resolver; the second asks DNS directly and shows the records; the third lists the routers your packets cross, hop by hop, with the round-trip time to each: watch the times jump when the packets cross an ocean.

## Quiz

## Daily challenge

## Before you go

Run the On your own computer commands for a site you use and note how many hops away it is. Tomorrow: **HTTP**, the protocol of the web: requests and responses, methods, status codes, headers, and what's really written on the wire.